# Sequence Transformer Training for ISL Dataset
**Upload this Notebook on Google Colab to use virtual GPU for Training**

In [ ]:
!pip install torch torchvision torchaudio
!pip install scikit-learn
!pip install mediapipe opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
# --- Setup & Unzip ---
import os
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import math

# Unzip the uploaded data
!unzip -q sequences.zip -d dataset/
print("Data unzipped successfully.")

Data unzipped successfully.


In [ ]:
# --- COLAB CELL 2: PyTorch Dataset ---
class ISLDataset(Dataset):
    def __init__(self, data_dir):
        self.data = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        for cls_name in self.classes:
            cls_dir = os.path.join(data_dir, cls_name)
            if not os.path.isdir(cls_dir): continue

            for file in os.listdir(cls_dir):
                if file.endswith('.npy'):
                    # Load the 30x126 sequence
                    seq = np.load(os.path.join(cls_dir, file))
                    self.data.append(seq)
                    self.labels.append(self.class_to_idx[cls_name])

        self.data = torch.FloatTensor(np.array(self.data))
        self.labels = torch.LongTensor(self.labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [ ]:
# --- COLAB CELL 3: The Transformer Model ---
class PositionalEncoding(nn.Module):
    """Injects positional information so the Transformer knows frame order."""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ISLTransformer(nn.Module):
    def __init__(self, num_classes, input_dim=126, d_model=128, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)

        # Classification head
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch, seq_len=30, input_dim=126)
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)

        # Take the output of the final time step
        x = x[:, -1, :]
        x = self.dropout(x)
        out = self.fc(x)
        return out

In [ ]:
# --- COLAB CELL 4: Training Pipeline ---
# 1. Prepare Data
full_dataset = ISLDataset('dataset/sequences')
num_classes = len(full_dataset.classes)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Setup Device & Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model = ISLTransformer(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 3. Training Loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == batch_y).sum().item()

    train_acc = 100 * correct / len(train_dataset)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}, Acc: {train_acc:.2f}%')

Training on: cuda
Epoch [10/50], Loss: 0.9769, Acc: 74.19%
Epoch [20/50], Loss: 0.2343, Acc: 91.94%
Epoch [30/50], Loss: 0.0527, Acc: 100.00%
Epoch [40/50], Loss: 0.0258, Acc: 98.39%
Epoch [50/50], Loss: 0.1573, Acc: 93.55%


In [ ]:
# --- COLAB CELL 5: Export Models ---
# Save the model weights
torch.save(model.state_dict(), 'transformer_isl.pth')

# Save the label mapping so your local script knows what classes are
with open('label_map.json', 'w') as f:
    json.dump(full_dataset.class_to_idx, f)

print("\nTraining Complete! Download 'transformer_isl.pth' and 'label_map.json' to your local machine.")


Training Complete! Download 'transformer_isl.pth' and 'label_map.json' to your local machine.
